# Witaj w generatorze checklist żeglarskich!

Naciśnij na górze przycisk "Run all". 

Następnie trzeba poczekać parę sekund, by zainstalowały się potrzebne pliki. 

Aplikacja jest gotowa, gdy poniżej pojawią się checkboxy do wyboru charakteru rejsu

In [31]:
# @title
import os

if not os.path.exists('sailing-checklist'):
    !git clone https://github.com/SailNCode/sailing-checklist.git
else:
    !cd sailing-checklist && git pull && cd ..

Already up to date.


In [32]:
# @title
%pip install -q ipywidgets


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Wybierz płeć:

In [33]:
# @title
import ipywidgets as widgets
from IPython.display import display

gender_types = {
    "Kobieta": "f",
    "Mężczyzna": "m"
}
gender_choice = widgets.RadioButtons(
    options=list(gender_types.keys()),
    value=list(gender_types.keys())[0]
)

display(gender_choice)

RadioButtons(options=('Kobieta', 'Mężczyzna'), value='Kobieta')

Wybierz charakter rejsu (można kilka):

In [ ]:
# @title

from pathlib import Path
DIRECTORY = Path("sailing-checklist/categories")
INDIVIDUAL_STEM = "prywatne"
ALWAYS_GET_NAME = "zawsze_zabrać"

filenames_to_labels = {}
for file in DIRECTORY.iterdir():
    filename = file.name
    stem = filename[:-5]  # strip .json
    label = " ".join(stem.split("_")).capitalize()
    filenames_to_labels[filename] = label

individual_of = {}
parents = {}

for filename in filenames_to_labels:
    stem = filename[:-5]
    if stem.endswith(f"_{INDIVIDUAL_STEM}"):
        parent_filename = stem[: -len(f"_{INDIVIDUAL_STEM}")] + ".json"
        individual_of[parent_filename] = filename
    else:
        parents[filename] = filenames_to_labels[filename]

# Build checkboxes
categories_choices = {}

for filename, label in parents.items():
    categories_choices[filename] = widgets.Checkbox(description=label)

    if filename in individual_of:
        ind_filename = individual_of[filename]
        parts = filenames_to_labels[ind_filename].split(" ")
        ind_label = parts[len(parts) - 1].capitalize()
        ind_checkbox = widgets.Checkbox(description=INDIVIDUAL_STEM.capitalize())
        ind_checkbox.layout = widgets.Layout(margin="0 0 0 30px")  # indent
        categories_choices[ind_filename] = ind_checkbox

categories_choices[f"{ALWAYS_GET_NAME}.json"].value = True

# Display parents followed immediately by their individual (if any)
display_order = []
for filename in parents:
    display_order.append(categories_choices[filename])
    if filename in individual_of:
        display_order.append(categories_choices[individual_of[filename]])

display(*display_order)

Checkbox(value=False, description='Zagraniczne')

Checkbox(value=False, description='Bojery')

Checkbox(value=False, description='Prywatne', layout=Layout(margin='0 0 0 30px'))

Checkbox(value=True, description='Zawsze zabrać')

Checkbox(value=False, description='Prywatne', layout=Layout(margin='0 0 0 30px'))

Checkbox(value=False, description='Mazury')

Checkbox(value=False, description='Ciepłe morze')

Wygeneruj checklistę żeglarską poniżej:

In [ ]:
# @title
import json
from pathlib import Path

import ipywidgets as widgets
from IPython.display import Javascript, display

SEPARATOR = "_"

def add_item(raw_item: str, category: str, target: dict, user_gender: str) -> None:
    parts = raw_item.split(SEPARATOR)

    if len(parts) == 1:
        if target.get(category) is None:
            target[category] = []
        if raw_item not in target[category]:
            target[category].append(raw_item)
        return

    item_name, item_gender = parts[0], parts[1]
    if user_gender == item_gender:
        if target.get(category) is None:
            target[category] = []
        if item_name not in target[category]:
            target[category].append(item_name)


def load_category(category: str) -> dict | None:
    try:
        with open(DIRECTORY / f"{category}", "r") as file:
            return json.load(file)
    except FileNotFoundError:
        print(f"Error: There is no '{category}' list.")
        return None


def format_output(data: dict[str, list[str]]) -> str:
    lines = []
    for group, items in data.items():
        lines.append(group)
        for item in items:
            lines.append(f"\t{item}")
    return "\n".join(lines)

def on_generate_clicked(_):
    user_gender = gender_types[gender_choice.value]
    joined_list: dict[str, list[str]] = {}

    chosen_categories = [
        key for key, checkbox in categories_choices.items() if checkbox.value
    ]

    for category in chosen_categories:
        data = load_category(category)
        if not data:
            continue

        for name, item in data.items():

            if isinstance(item, str):
                add_item(item, name, joined_list, user_gender)

            elif isinstance(item, list):
                for list_item in item:
                    add_item(list_item, name, joined_list, user_gender)

    text_output.value = format_output(joined_list)


def on_copy_clicked(_):
    display(
        Javascript(
            f"navigator.clipboard.writeText(`{text_output.value}`);"
        )
    )

generate_button = widgets.Button(
    description="Wygeneruj",
    button_style="success"
)
generate_button.on_click(on_generate_clicked)

copy_button = widgets.Button(
    description="Kopiuj",
    icon="copy",
)
copy_button.on_click(on_copy_clicked)

text_output = widgets.Textarea(
    value="",
    placeholder="Tu pojawi się lista...",
    layout=widgets.Layout(width="100%", height="200px"),
)

display(widgets.VBox([text_output, widgets.HBox([generate_button, copy_button])]))
